# Source Inventory Review

Inspection notebook for the pilot-case source inventory. This notebook reads the tracking CSV and summarizes coverage without adding or modifying source entries.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

pilot_cases = ["Western Roman Empire", "Maya", "Soviet Union"]

In [ ]:
project_root = Path.cwd()
source_inventory_path = project_root / "references" / "sources" / "source_inventory.csv"

if not source_inventory_path.exists():
    project_root = Path.cwd().parent
    source_inventory_path = project_root / "references" / "sources" / "source_inventory.csv"

if not source_inventory_path.exists():
    raise FileNotFoundError(f"Source inventory not found at {source_inventory_path}")

source_inventory_path

In [ ]:
df = pd.read_csv(source_inventory_path)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head()

## Source Counts By Case

In [ ]:
source_counts_by_case = (
    df["case_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .value_counts(dropna=True)
    .reindex(pilot_cases, fill_value=0)
    .rename_axis("case_name")
    .reset_index(name="source_count")
)

source_counts_by_case

## Empty Fields

In [ ]:
def is_blank_or_missing(series: pd.Series) -> pd.Series:
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        return series.isna() | series.fillna("").astype(str).str.strip().eq("")
    return series.isna()


empty_field_summary = pd.DataFrame(
    {
        "empty_count": [int(is_blank_or_missing(df[column]).sum()) for column in df.columns],
        "empty_pct": [float(is_blank_or_missing(df[column]).mean() * 100) if len(df) else 0.0 for column in df.columns],
    },
    index=df.columns,
).sort_values(["empty_count", "empty_pct"], ascending=False)

empty_field_summary